In [2]:
import gymnasium
import flappy_bird_gymnasium
import torch
from torch import nn
import torch.nn.functional as F
from collections import deque
import random
import itertools
import yaml
import random
from DQN import *
from ReplayMemory import *
import os
from datetime import datetime, timedelta

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
RUNS_DIR = 'runs'
os.makedirs(RUNS_DIR , exist_ok=True)

Training on: cpu


## agent class:
- have 2 functions other than the constructor
- optimize(self, mini_batch, policy_dqn, target_dqn): 
    - this calculate the target
    - then calculate the error
    - then do the back propagation and do the weights update

- run(self, is_training=True, render=False):
    - inializes the environmnet
    - instantiae the two networks
    - generate actions either from the policy of randomly 
    - store experince in the Reply Memory

In [7]:
class agent():

    def __init__(self, hyperparameter_set):
        with open('hyperparameters.yml', 'r') as file:
            all_hyperparameter_sets = yaml.safe_load(file)
            hyperparameters = all_hyperparameter_sets[hyperparameter_set]

        self.replay_memory_size = hyperparameters['replay_memory_size']
        self.mini_batch_size    = hyperparameters['mini_batch_size']
        self.epsilon_init       = hyperparameters['epsilon_init']
        self.epsilon_decay      = hyperparameters['epsilon_decay']
        self.epsilon_min        = hyperparameters['epsilon_min']
        self.env_id             = hyperparameters['env_id']
        self.learning_rate_a    = hyperparameters['learning_rate_a']
        self.discount_factor_g  = hyperparameters['discount_factor_g']
        self.network_sync_rate  = hyperparameters['network_sync_rate']
        self.hidden_layer_units = hyperparameters['hidden_layer_units']
        self.stop_on_reward     = hyperparameters['stop_on_reward']

        self.loss_fn = nn.MSELoss()
        self.optimizer = None

        self.LOG_FILE   = os.path.join(RUNS_DIR, f'{hyperparameter_set}.log')
        self.MODEL_FILE = os.path.join(RUNS_DIR, f'{hyperparameter_set}.pt')


    def optimize(self, mini_batch, policy_dqn, target_dqn): 
        pass
        
    
    def run(self, is_training=True, render=False):
        pass
        

- optimize(self, mini_batch, policy_dqn, target_dqn): 
    - this calculate the target
    - then calculate the error
    - then do the back propagation and do the weights update

In [10]:
def optimize(self, mini_batch, policy_dqn, target_dqn):
    states, actions, new_states, rewards, terminations = zip(*mini_batch)

    states       = torch.stack(states)
    actions      = torch.stack(actions)
    new_states   = torch.stack(new_states)
    rewards      = torch.stack(rewards)
    terminations = torch.tensor(terminations).float().to(device)
    
    with torch.no_grad():
        target_q = rewards + (1-terminations) * self.discount_factor_g * target_dqn(new_states).max(dim=1)[0]

    current_q = policy_dqn(states).gather(dim=1, index=actions.unsqueeze(dim=1)).squeeze()
    loss = self.loss_fn(current_q, target_q)

    ##optimization model
    self.optimizer.zero_grad() #Clear Gradients of the last step
    loss.backward()            #compute gradients
    self.optimizer.step()      # update weights

- run(self, is_training=True, render=False):
    - inializes the environmnet
    - instantiae the two networks
    - generate actions either from the policy of randomly 
    - store experince in the Reply Memory

In [15]:
def run(self, is_training=True, render=False):
    

    env = gymnasium.make(self.env_id, render_mode='human' if render else None)

    num_of_actions = env.action_space.n
    num_of_states = env.observation_space.shape[0]
    rewards_per_episode = []
    epsilon_history = []
    policy_dqn = DQN(num_of_states, num_of_actions, hidden_dim = self.hidden_layer_units).to(device)

   
    if is_training:
        memory     = ReplayMemory(self.replay_memory_size) #instantiate memory to store experiences
        target_dqn = DQN(num_of_states, num_of_actions, hidden_dim = self.hidden_layer_units).to(device)
        target_dqn.load_state_dict(policy_dqn.state_dict()) #clone the policy_dqn
        epsilon    = self.epsilon_init
        #count the no. of steps to update the target net
        step_count = 0
        self.optimizer = torch.optim.Adam(policy_dqn.parameters(), lr=self.learning_rate_a)
        best_reward = -9999999

    else:
        #load the trained model
        policy_dqn.load_state_dict(torch.load(self.MODEL_FILE))
        policy_dqn.eval()

#################################################### start episode ##################################################        
    for episodes in itertools.count(): #each episode runs until a termination state
        state, _ = env.reset()
        state = torch.tensor(state, dtype=torch.float, device=device)
        terminated = False
        episode_reward = 0.0 
        no_of_exeplorations_per_episode = 0
        no_of_exeploitations_per_episode = 0
        while not terminated and episode_reward < self.stop_on_reward:
            if is_training and random.random()<epsilon:
                action = env.action_space.sample() #exploration
                action = torch.tensor(action, dtype=torch.int64, device=device)
                no_of_exeplorations_per_episode +=1
            else:
                with torch.no_grad(): #just the forward path
                    action = policy_dqn(state.unsqueeze(dim=0)).squeeze().argmax() #exploitation
                #argmax used to return the index of the max no. in the output
                no_of_exeploitations_per_episode +=1
                
            new_state, reward, terminated, _, info = env.step(action.item())
            new_state = torch.tensor(new_state, dtype=torch.float, device=device)
            reward = torch.tensor(reward, dtype=torch.float, device=device)
            episode_reward += reward ##accumulate the reward
            
            if is_training:
                memory.append((state, action, new_state, reward, terminated))
                step_count +=1
                
            state = new_state
        #################################################### end episode ##################################################

        if episodes%1000 == 0:
            print(f"at episode {episodes} total no. of explore { no_of_exeplorations_per_episode}, total no. of exploite { no_of_exeploitations_per_episode} and the reward {episode_reward}")
        rewards_per_episode.append(episode_reward)

        if is_training:
            if episode_reward>best_reward:
                torch.save(policy_dqn.state_dict(), self.MODEL_FILE)
                best_reward = episode_reward
                log_message = f"at time {datetime.now()} New best reward: {best_reward} at episode: {episodes}"
                print(log_message)
                with open(self.LOG_FILE,'a') as file:
                    file.write(log_message + '\n')

            #if enough experience have been collected
            if len(memory) > self.mini_batch_size:
                
                mini_batch = memory.sample(self.mini_batch_size)
                self.optimize(mini_batch, policy_dqn, target_dqn)
                epsilon = max(self.epsilon_min, self.epsilon_decay*epsilon)
                epsilon_history.append(epsilon)
                
                #update the target network weights
                if step_count> self.network_sync_rate:
                    target_dqn.load_state_dict(policy_dqn.state_dict())
                    step_count = 0


        else:
            if episode_reward> 3000:
                print(f"win the game")
                break
                

In [17]:
agent.run = run
agent.optimize = optimitze 

NameError: name 'optimitze' is not defined

In [55]:
if __name__ == '__main__':
    agent = agent('cartpole1')
    agent.run(is_training=True, render=False)

at episode 0 total no. of explore 12, total no. of exploite 0 and the reward 12.0
at time 2024-12-31 13:45:01.915843 New best reward: 12.0 at episode: 0
at time 2024-12-31 13:45:01.921360 New best reward: 22.0 at episode: 1
at time 2024-12-31 13:45:01.927376 New best reward: 41.0 at episode: 4
at time 2024-12-31 13:45:01.941784 New best reward: 42.0 at episode: 11
at time 2024-12-31 13:45:01.966152 New best reward: 53.0 at episode: 23
at time 2024-12-31 13:45:02.063555 New best reward: 73.0 at episode: 77
at time 2024-12-31 13:45:02.229939 New best reward: 82.0 at episode: 161
at time 2024-12-31 13:45:02.542568 New best reward: 129.0 at episode: 306
at time 2024-12-31 13:45:03.102734 New best reward: 190.0 at episode: 540
at time 2024-12-31 13:45:03.993937 New best reward: 205.0 at episode: 799
at episode 1000 total no. of explore 33, total no. of exploite 12 and the reward 45.0
at time 2024-12-31 13:45:05.362081 New best reward: 258.0 at episode: 1024
at time 2024-12-31 13:45:05.42271

KeyboardInterrupt: 

In [19]:
if __name__ == '__main__':
    agent2 = agent('cartpole1')
    agent2.run(is_training=False, render=True)

C:\Users\user\AppData\Local\Temp\ipykernel_19540\2430774654.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  policy_dqn.load_state_dict(torch.load(self.MODEL_FILE))


KeyboardInterrupt: 

In [23]:
states = [torch.tensor([1, 2]), torch.tensor([3, 4])]
print(states)
states = torch.stack(states)
print(states)

tensor([[1, 2],
        [3, 4]])